# Imports 

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configuration visuelle
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

# Chargement et Exploration des Données

In [ ]:

url = "https://perso.esiee.fr/~kachourr/RK_DataScience/iris.data"
colonnes = ["sepal_length", "sepal_width", "petal_length", "petal_width", "Class"]
df_iris = pd.read_csv(url, names=colonnes)

print("Aperçu des données :")
display(df_iris.head())

print("\nInformations sur le dataset :")
display(df_iris.info())
print()

# Visualisation des distributions
plt.figure(figsize=(10, 6))
sns.pairplot(df_iris, hue="Class", palette="plasma", markers=["o", "s", "D"])
plt.suptitle("Distribution des caractéristiques par espèce", y=1.02)
plt.show()

# Préparation des Données et Séparation

In [ ]:

# Séparation des features (X) et de la cible (y)
X = df_iris.drop('Class', axis=1)
y = df_iris['Class']

# Encodage de la variable cible
classes_mapping = {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}
y_encoded = y.map(classes_mapping)

# Séparation en jeu d'entraînement 
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"Taille du jeu d'entraînement : {X_train.shape}")
print(f"Taille du jeu de test : {X_test.shape}")

# Modélisation Supervisée

In [ ]:

# Création d'un pipeline 
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

# Entraînement du modèle
model_pipeline.fit(X_train, y_train)

# Prédictions sur le jeu de test
y_pred = model_pipeline.predict(X_test)

# Évaluation des performances
print("Matrice de confusion :")
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, cmap="Blues", fmt='g', 
            xticklabels=classes_mapping.keys(), yticklabels=classes_mapping.keys())
plt.show()

print("\nRapport de classification :")
print(classification_report(y_test, y_pred, target_names=classes_mapping.keys()))

# Analyse Non Supervisée : Réduction de dimension (ACP) & Clustering (KMeans)

In [ ]:

# Normalisation globale pour l'analyse visuelle
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 1. Réduction de dimension (ACP)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# 2. Clustering (K-Means)
kmeans = KMeans(n_clusters=3, random_state=223, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Visualisation combinée
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot ACP avec les Vraies classes
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y_encoded, cmap='plasma', edgecolor='k', s=60)
ax1.set_title("Projection ACP (Vraies Espèces)")
ax1.set_xlabel("Composante Principale 1")
ax1.set_ylabel("Composante Principale 2")

# Plot ACP avec les labels K-Means
scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', edgecolor='k', s=60)
ax2.set_title("Projection ACP (Clusters K-Means)")
ax2.set_xlabel("Composante Principale 1")

plt.show()

# Déploiement : Interface Gradio

In [ ]:

# Dictionnaire inverse pour repasser des nombres aux noms des espèces
inverse_mapping = {v: k for k, v in classes_mapping.items()}

def predict_iris(sepal_l, sepal_w, petal_l, petal_w):
    # Création d'un dataframe avec les noms de colonnes attendus par le modèle
    features = pd.DataFrame(
        [[sepal_l, sepal_w, petal_l, petal_w]], 
        columns=X.columns
    )
    # Prédiction via le pipeline ( gère automatiquement le StandardScaler)
    prediction_idx = model_pipeline.predict(features)[0]
    return inverse_mapping[prediction_idx]

# Création de l'interface
interface = gr.Interface(
    fn=predict_iris, 
    inputs=[
        gr.Slider(minimum=4.0, maximum=8.0, step=0.1, label="Longueur du Sépale (cm)"),
        gr.Slider(minimum=2.0, maximum=4.5, step=0.1, label="Largeur du Sépale (cm)"),
        gr.Slider(minimum=1.0, maximum=7.0, step=0.1, label="Longueur du Pétale (cm)"),
        gr.Slider(minimum=0.1, maximum=2.5, step=0.1, label="Largeur du Pétale (cm)")
    ],
    outputs=gr.Textbox(label="Espèce prédite"),
    title="Classificateur de Fleurs d'Iris",
    description="Entrez les dimensions de la fleur pour qu'un modèle de Machine Learning Random Forest prédise son espèce."
)

# Lancement 
if __name__ == "__main__":
    interface.launch(share=False)